# 🎯 Week 5 — Product Recommendation System
**Smart E-Commerce Analytics Platform**

**Dataset:** Flipkart Product Reviews (Kaggle) → `flipkart_clean.csv`
- 189,869 product reviews
- Columns: product_name, category, price, rating, review_text, summary

**This notebook covers:**
1. Load & inspect Flipkart dataset
2. Data cleaning & preprocessing
3. Popularity-based recommendations
4. Content-based filtering (TF-IDF on review text)
5. Cosine similarity matrix
6. Recommendation examples & evaluation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
print('Libraries loaded ✅')

## 1. Load & Inspect Raw Flipkart Data

In [ ]:
# Load raw file
raw = pd.read_csv('../data/flipkart/flipkart_product.csv', encoding='latin-1')
print(f'Raw shape: {raw.shape}')
print(f'Columns: {list(raw.columns)}')
display(raw.head(3))
print('\nMissing values:')
print(raw.isnull().sum())
print(f'\nRating range: {raw["Rate"].unique()[:10]}')

## 2. Load Cleaned Dataset

In [ ]:
df = pd.read_csv('../data/flipkart_clean.csv')
print(f'Cleaned shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Unique products: {df["product_name"].nunique():,}')
print(f'Unique categories: {df["category"].nunique()}')
print(f'Rating range: {df["rating"].min()} – {df["rating"].max()}')
print(f'Avg rating: {df["rating"].mean():.2f}')
display(df.head(3))

## 3. Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Rating distribution
rating_cnt = df['rating'].value_counts().sort_index()
colors = ['#d73027','#f46d43','#fdae61','#a6d96a','#1a9850']
rating_cnt.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Top 10 categories by review count
cat_cnt = df['category'].value_counts().head(10)
cat_cnt.plot(kind='barh', ax=axes[1], color=sns.color_palette('Blues_r', 10))
axes[1].set_title('Top 10 Categories by Reviews')
axes[1].set_xlabel('Review Count')
axes[1].invert_yaxis()

# Price distribution (capped)
price_data = df['price'].dropna()
price_cap  = price_data.quantile(0.95)
axes[2].hist(price_data.clip(upper=price_cap), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[2].set_title(f'Price Distribution (95th pct cap)')
axes[2].set_xlabel('Price (₹)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../report/flipkart_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Popularity-Based Recommendations

In [ ]:
# Aggregate per product
pop = (df.groupby(['product_name', 'category'])
       .agg(
           review_count = ('rating', 'count'),
           avg_rating   = ('rating', 'mean'),
           avg_price    = ('price',  'mean'),
       ).reset_index())
pop['avg_rating'] = pop['avg_rating'].round(2)
pop['avg_price']  = pop['avg_price'].round(2)

# Popularity score = 60% review volume + 40% avg rating (both normalized)
sc = MinMaxScaler()
pop[['cnt_n', 'rat_n']] = sc.fit_transform(pop[['review_count', 'avg_rating']])
pop['pop_score'] = (0.6 * pop['cnt_n'] + 0.4 * pop['rat_n']).round(4)
pop = pop.sort_values('pop_score', ascending=False).reset_index(drop=True)

print('Top 10 Most Popular Products:')
display(pop[['product_name','category','review_count','avg_rating','avg_price','pop_score']].head(10))

In [ ]:
# Visualize top 10
top10 = pop.head(10)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh([n[:40] for n in top10['product_name']], top10['pop_score'],
        color=sns.color_palette('Blues_r', 10))
ax.set_title('Top 10 Products by Popularity Score')
ax.set_xlabel('Popularity Score')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../report/top_products_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

def get_popular_recommendations(category=None, n=5):
    """Get top-n popular products, optionally filtered by category."""
    result = pop.copy()
    if category:
        result = result[result['category'].str.lower() == category.lower()]
    return result[['product_name','category','review_count','avg_rating','pop_score']].head(n)

print('\nTop 5 overall:')
display(get_popular_recommendations(n=5))

## 5. Content-Based Filtering (TF-IDF)

In [ ]:
# Aggregate reviews per product
prod = (df.groupby('product_name')
        .agg(
            category     = ('category',    'first'),
            avg_rating   = ('rating',      'mean'),
            avg_price    = ('price',       'mean'),
            review_count = ('rating',      'count'),
            text         = ('review_text', lambda x: ' '.join(x.dropna().astype(str)[:10]))
        ).reset_index())
prod['avg_rating'] = prod['avg_rating'].round(2)

print(f'Unique products for content model: {len(prod):,}')
print(f'Sample text (first product):')
print(prod['text'].iloc[0][:200])

In [ ]:
# Build TF-IDF matrix
tfidf = TfidfVectorizer(max_features=3000, stop_words='english', min_df=2)
tfidf_mat = tfidf.fit_transform(prod['text'].fillna(''))

print(f'TF-IDF matrix shape: {tfidf_mat.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,}')

# Compute cosine similarity
sim_matrix = cosine_similarity(tfidf_mat, tfidf_mat)
sim_df = pd.DataFrame(sim_matrix, index=prod['product_name'], columns=prod['product_name'])

print(f'Similarity matrix shape: {sim_df.shape}')
print('\nSample similarity scores (first product vs others):')
print(sim_df.iloc[0].sort_values(ascending=False).head(5))

## 6. Recommendation Function & Examples

In [ ]:
def get_content_recommendations(product_name, n=5):
    """Get top-n similar products using TF-IDF cosine similarity."""
    if product_name not in sim_df.index:
        print(f'Product not found: {product_name}')
        return None
    sim_scores = sim_df[product_name].drop(index=product_name, errors='ignore')
    top_similar = sim_scores.sort_values(ascending=False).head(n)
    recs = prod[prod['product_name'].isin(top_similar.index)].copy()
    recs['similarity'] = recs['product_name'].map(top_similar).round(4)
    return recs[['product_name','category','avg_rating','avg_price','similarity']].sort_values('similarity', ascending=False)

# Example: get recommendations for the most popular product
top_product = pop.iloc[0]['product_name']
print(f'Recommendations for: "{top_product[:60]}"')
display(get_content_recommendations(top_product, n=5))

In [ ]:
# Similarity heatmap for top 8 products
top8 = pop.head(8)['product_name'].tolist()
top8 = [p for p in top8 if p in sim_df.index]

sub = sim_df.loc[top8, top8]
labels = [n[:25] for n in sub.index]
sub.index = labels
sub.columns = labels

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sub, annot=True, fmt='.2f', cmap='YlOrRd',
            ax=ax, linewidths=0.5, square=True)
ax.set_title('Product Similarity Heatmap (TF-IDF Cosine)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../report/similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Recommendation System Complete!')
print(f'Products indexed: {len(prod):,}')
print(f'Total reviews: {len(df):,}')